# Week 10: SFT Dataset & LoRA Training

# Requirements: pip install torch transformers accelerate peft trl bitsandbytes datasets numpy pandas

# ⚠️ REQUIRES: GPU (Colab free tier works) + internet

This notebook builds a ~300-example SFT dataset in **chat-template** format from two
ZoroLogistics tasks, bill-of-lading extraction (`data.bol_samples()`) and ticket triage
(`data.support_tickets()`), then trains a **LoRA/QLoRA** adapter with TRL's
`SFTTrainer` on a small base model, logs the loss curve, and saves the adapter. The
dataset cells are CPU-only; the training cells need a GPU.


## 0. Setup: repo root on the path + seeded RNG


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root

import os, json, time, math
import numpy as np
import pandas as pd

from zoro import data

SEED = 42
np.random.seed(SEED)

try:
    import torch
    from datasets import Dataset
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from trl import SFTTrainer
    _TRAIN_OK = True
except Exception as _e:  # pragma: no cover
    torch = None
    _TRAIN_OK = False
    print("training stack not installed:", _e)

if torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

print("imports ok, training stack:", _TRAIN_OK)


## 1. Why fine-tune at all? (Ng's selection order)

Fine-tuning is the **last** lever: prompt with an eval → RAG → agentic workflow →
fine-tune. You reach for it only when the base model + context cannot pass your eval on a
specific, high-volume task. The two tasks here are exactly that kind: a strict JSON
output format (BoL extraction) and a domain routing style (ticket triage) that a base
model gets only partly right.


In [ ]:
def cuda_available():
    try:
        return torch.cuda.is_available()
    except Exception:
        return False

HAS_CUDA = cuda_available()
print("CUDA available:", HAS_CUDA, "(training cells need it)")


## 2. Build the SFT dataset: part (a) BoL extraction

Each example is a `messages` array (system / user / assistant) matching the chat
template. The assistant answer is the **ground-truth JSON** of `bol["fields"]`, this is
the label the model learns to imitate. 150 examples.


In [ ]:
bols = data.bol_samples(n=150, seed=5)

EXTRACTION_SYSTEM = (
    "You are ZoroLogistics' document-extraction assistant. Given a bill of lading, "
    "extract the structured fields and return ONLY valid JSON with keys: shipper, "
    "consignee, port_of_loading, port_of_discharge, commodity, quantity, "
    "gross_weight_kg, declared_value_usd, freight_terms, date_of_issue."
)

def extraction_messages(bol):
    return {
        "messages": [
            {"role": "system", "content": EXTRACTION_SYSTEM},
            {"role": "user", "content": f"Extract the fields from this bill of lading:\n{bol['text']}"},
            {"role": "assistant", "content": json.dumps(bol["fields"], sort_keys=True)},
        ]
    }

extraction_examples = [extraction_messages(b) for b in bols]
print("extraction examples:", len(extraction_examples))
print("sample assistant output:", extraction_examples[0]["messages"][-1]["content"][:120], "...")


## 3. Build the SFT dataset: part (b) ticket triage

Same format, different task: the assistant answer is the ground-truth `category`. 150
more examples, for ~300 total.


In [ ]:
tickets = data.support_tickets(n=150, seed=99)

TRIAGE_SYSTEM = (
    "You are ZoroLogistics' support-router. Classify each ticket into exactly one "
    "category: tracking, damage, refund, documents, customs, or billing. Reply with "
    "only the category word."
)

def triage_messages(row):
    return {
        "messages": [
            {"role": "system", "content": TRIAGE_SYSTEM},
            {"role": "user", "content": row["text"]},
            {"role": "assistant", "content": row["category"]},
        ]
    }

triage_examples = [triage_messages(r) for _, r in tickets.iterrows()]
print("triage examples:", len(triage_examples))


## 4. Combine, save, and sanity-check

We write the dataset as JSONL (one chat-template object per line) so the same file feeds
TRL now and can be versioned later. A small model internalizes whatever pattern dominates
the data, so a quick length/stats pass is worth it before training.


In [ ]:
all_examples = extraction_examples + triage_examples
print("total SFT examples:", len(all_examples))

out_dir = pathlib.Path("sft_dataset")
out_dir.mkdir(exist_ok=True)
sft_path = out_dir / "sft_train.jsonl"
with open(sft_path, "w", encoding="utf-8") as f:
    for ex in all_examples:
        f.write(json.dumps(ex) + "\n")
print("saved", sft_path)

lens = [sum(len(m["content"].split()) for m in ex["messages"]) for ex in all_examples]
print(f"mean words/example: {np.mean(lens):.1f} | min {min(lens)} | max {max(lens)}")


## 5. Cost / time estimate (before you train)

- **Model:** `Qwen/Qwen2.5-1.5B-Instruct`, QLoRA 4-bit NF4 (`r=16`) → ~0.75 GB weights.
- **Data:** 300 examples, ~40 to 80 tokens each; 1 epoch.
- **Compute:** effective batch 8 (batch 2 × grad-accum 4) → ~38 optimizer steps.
- **VRAM:** fits a free Colab **T4 (16 GB)** or an 8 GB consumer GPU.
- **Time:** a few minutes on a T4. **Cost:** $0 on Colab free tier; a few cents on rented GPU.
- **No GPU?** run the dataset cells (CPU-only) and record the stats; the trainer skips.


## 6. Tokenizer + chat template

We load the tokenizer and make sure it has a `pad_token` (Qwen2.5 uses `eos_token` as
pad) so batched training and generation behave.


In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = None
if _TRAIN_OK:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("tokenizer loaded; pad_token is eos:", tokenizer.pad_token == tokenizer.eos_token)
else:
    print("Skipping tokenizer, training stack missing.")


## 7. LoRA config

LoRA freezes the base and trains low-rank matrices (`ΔW = BA`) injected into the
attention/MLP projections. `r=16`, `alpha=32` is a common starting point; the target
modules are Qwen's attention (q/k/v/o) and MLP (gate/up/down) projections.


In [ ]:
LORA_CONFIG = dict(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
print("LoRA config:", LORA_CONFIG)


## 8. Load the base model

**QLoRA** on CUDA: 4-bit NF4 base (bitsandbytes) + LoRA on top, that is what makes a
1.5B model trivially fit, and the same recipe scales to 7 to 8B on a single card. On Apple
Silicon we fall back to FP16 plain LoRA; with no accelerator we skip.


In [ ]:
model = None
if not _TRAIN_OK:
    print("Skipping model load, training stack missing.")
elif HAS_CUDA:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
    )
    model = prepare_model_for_kbit_training(model)
    print("Loaded 4-bit base (QLoRA)")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.float16, device_map="mps", trust_remote_code=True
    )
    print("Loaded FP16 base on MPS (plain LoRA)")
else:
    print("No CUDA/MPS, skipping model load; run dataset cells only.")


In [ ]:
adapter_dir = pathlib.Path("lora_adapter")
if model is not None:
    peft_config = LoraConfig(**LORA_CONFIG)
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
else:
    print("No model to wrap.")


## 9. Training arguments

Small model, small data → 1 epoch is enough to learn the shape without overfitting.
`fp16` is set only when CUDA is present. `report_to=[]` keeps the lab self-contained
(no external tracker needed; swap in MLflow later).


In [ ]:
train_args = TrainingArguments(
    output_dir=str(out_dir / "checkpoints"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    fp16=HAS_CUDA,
    report_to=[],
    seed=SEED,
)

steps_per_epoch = math.ceil(len(all_examples) / (2 * 4))
print(f"examples={len(all_examples)}, effective batch=8, ~{steps_per_epoch} optimizer steps/epoch")
print("estimated wall time on a T4: a few minutes (1 epoch)")


## 10. Train (the only long cell)

`SFTTrainer` wires the dataset, tokenizer, and LoRA model together and runs the loop. We
capture the loss history so we can read the curve afterward.


In [ ]:
train_losses = []
if model is not None and tokenizer is not None:
    train_ds = Dataset.from_list(all_examples)

    trainer = SFTTrainer(
        model=model,
        args=train_args,
        train_dataset=train_ds,
        tokenizer=tokenizer,
        max_seq_length=512,
        packing=False,
    )
    result = trainer.train()

    for entry in trainer.state.log_history:
        if "loss" in entry:
            train_losses.append(entry["loss"])
    final_loss = train_losses[-1] if train_losses else float("nan")
    print("training complete; logged losses:", len(train_losses))
else:
    final_loss = float("nan")
    print("Skipping training, no model/tokenizer (GPU required).")


## 11. Read the loss curve + save the adapter

A clean run shows loss falling then flattening. If it spikes or flatlines immediately,
something is off (LR too high, data not tokenizing). We save the adapter (not the base)
it is a few MB and is the deployable artifact.


In [ ]:
if train_losses:
    for i, l in enumerate(train_losses):
        print(f"step {i * 10:>4}: loss {l:.4f}")
    print("loss curve:", " ".join(f"{l:.2f}" for l in train_losses))

    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print("adapter saved to", adapter_dir)
else:
    print("no loss curve, training did not run.")


## 12. Sanity check: generate one extraction

A quick greedy decode on a fresh bill of lading shows whether the adapter learned the
JSON shape at all. This is a *smoke test*, not the eval, the real before/after numbers
come in notebook 02.


In [ ]:
test_bol = data.bol_samples(n=1, seed=123)[0]
prompt = f"{EXTRACTION_SYSTEM}\n\nExtract the fields from this bill of lading:\n{test_bol['text']}"

if model is not None and tokenizer is not None:
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        gen = model.generate(**enc, max_new_tokens=200, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    out = tokenizer.decode(gen[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    print("generated extraction:\n", out)
else:
    print("Skipping generation test, no model loaded.")


## 13. Takeaway

SFT is imitation learning: the adapter learned *what the data showed it*. Quality beats
quantity, 300 clean on-task examples beat thousands of noisy ones. The loss curve tells
you training happened; only the before/after eval (notebook 02) tells you it *helped*.


In [ ]:
# FINAL number: final training loss (or -1 if training did not run, dataset still built).
score = final_loss if (train_losses and final_loss == final_loss) else -1.0
print(f"SFT_FINAL_LOSS={score:.4f}")
print(f"SFT_EXAMPLES={len(all_examples)}")
